# Jacobik Data Preparation Guide

Convert custom user structures (XYZ files + SMILES) into the validated tmQM-format dataset structure expected by Jacobik loaders.

In [ ]:
import json
import sys
from pathlib import Path

# Walk upwards to locate repository root containing src/
repo = Path.cwd().resolve()
while repo != repo.parent and not (repo / "src").exists():
    repo = repo.parent

if str(repo / "src") not in sys.path:
    sys.path.insert(0, str(repo / "src"))
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

print(f"Jacobik repository root: {repo}")
import jacobik
from interface.ligand_library import DEBUG_COMPLEX_OJIMIF
from interface.prepare_data import prepare_dataset, parse_xyz, validate_structure
from fit.parser import TMQMParser


## 1. Generate a Demo Raw XYZ & Manifest From `DEBUG_COMPLEX_OJIMIF`

In [ ]:
raw_dir = Path("/tmp/prepared_demo_raw")
raw_dir.mkdir(parents=True, exist_ok=True)
demo_res = jacobik.generate(*DEBUG_COMPLEX_OJIMIF, seed=11, n_conformers=1, embed="etkdg", n_ik_steps=10, device="cpu")
raw_xyz = raw_dir / "OJIMIF.xyz"
demo_res.best.write_xyz(raw_xyz)

symbols, coords, _ = parse_xyz(raw_xyz)
smiles = "CCSC#N->[Pt+2](<-[Cl-])(<-[Cl-])<-N#CSCC"
valid, msgs, metal_idx = validate_structure(symbols, coords, "OJIMIF", smiles)
for msg in msgs:
    print(msg)
assert valid
print(f"Structure is valid with {len(symbols)} atoms and metal center at index {metal_idx}.")


## 2. Generate tmQM-Format Dataset Directory

In [ ]:
manifest = [
    {
        "csd_code": "OJIMIF",
        "xyz_file": "OJIMIF.xyz",
        "smiles": smiles,
        "charge": DEBUG_COMPLEX_OJIMIF.charge,
        "spin": 0,
    }
]
(raw_dir / "inputs.json").write_text(json.dumps(manifest, indent=2))
out_dir = Path("/tmp/prepared_demo")
prepare_dataset(manifest, out_dir, split_name="demo", base_dir=raw_dir)
print(f"Prepared files in {out_dir}:")
for f in sorted(out_dir.rglob("*")):
    if f.is_file():
        print(f"  - {f.relative_to(out_dir)} ({f.stat().st_size} bytes)")


## 3. Verify Parsing with TMQMParser

In [ ]:
parser = TMQMParser(out_dir)
mols = list(parser.stream_split("demo"))
for m in mols:
    print(f"Read {m.csd_code}: Metal {m.metal_symbol}, Atoms: {len(m.atoms)}")
assert len(mols) == len(manifest)
print("Verification successful!")
